In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

def clamp(value, low, high):
    return np.clip(value, low, high)

def add_noise(value, noise_scale, low=None, high=None):
    noisy = value + np.random.normal(0, noise_scale)
    if low is not None and high is not None:
        noisy = clamp(noisy, low, high)
    return noisy

def normalize_three(a, b, c):
    a = max(a, 0.1)
    b = max(b, 0.1)
    c = max(c, 0.1)
    total = a + b + c
    return a / total * 100, b / total * 100, c / total * 100

def assign_risk_score(row):
    score = 0
    score += (100 - row['lichen_coverage_percent']) * 0.25
    score += row['crustose_percent'] * 0.10
    score += (100 - row['fruticose_percent']) * 0.08
    score += row['nitrophyte_abundance'] * 0.10
    score += (100 - row['acidophyte_abundance']) * 0.08
    score += row['chlorosis_index'] * 35
    score += row['iron_ppm'] * 2.2
    score += row['copper_ppm'] * 2.8
    score += row['lead_ppm'] * 9.0
    score += row['zinc_ppm'] * 2.4
    score += row['manganese_ppm'] * 2.2
    score += row['no2_level'] * 0.45
    score += row['so2_level'] * 0.75
    score += row['aqi'] * 0.20
    score += np.random.normal(0, 4)
    return score

def score_to_label(score):
    if score < 58:
        return "Low"
    elif score < 82:
        return "Moderate"
    elif score < 108:
        return "High"
    else:
        return "Severe"

def generate_realistic_site(site_profile):
    if site_profile == 'wilderness':
        base = {'lichen_coverage_percent': np.random.normal(70, 10), 'crustose_percent': np.random.normal(25, 8), 'foliose_percent': np.random.normal(35, 8), 'fruticose_percent': np.random.normal(40, 10), 'nitrophyte_abundance': np.random.normal(20, 10), 'acidophyte_abundance': np.random.normal(70, 10), 'chlorosis_index': np.random.normal(0.12, 0.05), 'iron_ppm': np.random.normal(0.4, 0.2), 'copper_ppm': np.random.normal(0.2, 0.1), 'lead_ppm': np.random.normal(0.05, 0.03), 'zinc_ppm': np.random.normal(0.4, 0.2), 'manganese_ppm': np.random.normal(0.3, 0.2), 'no2_level': np.random.normal(5, 3), 'so2_level': np.random.normal(1.5, 1), 'aqi': np.random.normal(30, 10), "fluorine_estimate_ppb": np.random.normal(0.03, 0.02)}
    elif site_profile == 'suburban':
        base = {'lichen_coverage_percent': np.random.normal(50, 10), 'crustose_percent': np.random.normal(40, 10), 'foliose_percent': np.random.normal(35, 8), 'fruticose_percent': np.random.normal(25, 8), 'nitrophyte_abundance': np.random.normal(45, 12), 'acidophyte_abundance': np.random.normal(45, 12), 'chlorosis_index': np.random.normal(0.26, 0.06), 'iron_ppm': np.random.normal(1.0, 0.5), 'copper_ppm': np.random.normal(1.2, 0.6), 'lead_ppm': np.random.normal(0.15, 0.1), 'zinc_ppm': np.random.normal(1.4, 0.7), 'manganese_ppm': np.random.normal(0.8, 0.5), 'no2_level': np.random.normal(12, 5), 'so2_level': np.random.normal(3, 2), 'aqi': np.random.normal(55, 10), "fluorine_estimate_ppb": np.random.normal(0.10, 0.05)}
    elif site_profile == "traffic":
        base = {"lichen_coverage_percent": np.random.normal(28, 11), "crustose_percent": np.random.normal(58, 12), "foliose_percent": np.random.normal(29, 10), "fruticose_percent": np.random.normal(13, 7), "nitrophyte_abundance": np.random.normal(64, 12), "acidophyte_abundance": np.random.normal(29, 11), "chlorosis_index": np.random.normal(0.36, 0.08), "iron_ppm": np.random.normal(1.6, 0.7), "copper_ppm": np.random.normal(3.4, 1.0), "lead_ppm": np.random.normal(0.32, 0.12), "zinc_ppm": np.random.normal(3.8, 1.0), "manganese_ppm": np.random.normal(1.4, 0.6), "no2_level": np.random.normal(23, 6), "so2_level": np.random.normal(5, 2), "aqi": np.random.normal(72, 14), "fluorine_estimate_ppb": np.random.normal(0.18, 0.08)}
    elif site_profile == "industrial":
        base = {"lichen_coverage_percent": np.random.normal(23, 10), "crustose_percent": np.random.normal(64, 11), "foliose_percent": np.random.normal(27, 9), "fruticose_percent": np.random.normal(9, 5), "nitrophyte_abundance": np.random.normal(62, 12), "acidophyte_abundance": np.random.normal(28, 10), "chlorosis_index": np.random.normal(0.42, 0.09), "iron_ppm": np.random.normal(4.0, 1.1), "copper_ppm": np.random.normal(1.8, 0.7), "lead_ppm": np.random.normal(0.42, 0.15), "zinc_ppm": np.random.normal(2.0, 0.8), "manganese_ppm": np.random.normal(3.8, 1.1), "no2_level": np.random.normal(21, 6), "so2_level": np.random.normal(9, 3), "aqi": np.random.normal(82, 16), "fluorine_estimate_ppb": np.random.normal(0.45, 0.15)}
    elif site_profile == "severe_mixed":
        base = {"lichen_coverage_percent": np.random.normal(6, 5), "crustose_percent": np.random.normal(82, 8), "foliose_percent": np.random.normal(15, 7), "fruticose_percent": np.random.normal(3, 3), "nitrophyte_abundance": np.random.normal(82, 9), "acidophyte_abundance": np.random.normal(10, 6), "chlorosis_index": np.random.normal(0.68, 0.08), "iron_ppm": np.random.normal(5.8, 1.2), "copper_ppm": np.random.normal(4.4, 1.1), "lead_ppm": np.random.normal(0.85, 0.18), "zinc_ppm": np.random.normal(5.5, 1.1), "manganese_ppm": np.random.normal(5.4, 1.1), "no2_level": np.random.normal(35, 7), "so2_level": np.random.normal(22, 5), "aqi": np.random.normal(135, 22), "fluorine_estimate_ppb": np.random.normal(0.85, 0.25)}

    base['lichen_coverage_percent'] = clamp(base['lichen_coverage_percent'], 0, 100)
    crustose, foliose, fruticose = normalize_three(clamp(base['crustose_percent'], 0, 100), clamp(base['foliose_percent'], 0, 100), clamp(base['fruticose_percent'], 0, 100))
    base['crustose_percent'], base['foliose_percent'], base['fruticose_percent'] = crustose, foliose, fruticose
    base['nitrophyte_abundance'], base['acidophyte_abundance'], base['chlorosis_index'] = clamp(base['nitrophyte_abundance'], 0, 100), clamp(base['acidophyte_abundance'], 0, 100), clamp(base['chlorosis_index'], 0, 1)
    for m in ['iron_ppm', 'copper_ppm', 'lead_ppm', 'zinc_ppm', 'manganese_ppm']: base[m] = clamp(base[m], 0, 10)
    base['no2_level'], base['so2_level'], base['aqi'] = clamp(base['no2_level'], 0, 60), clamp(base['so2_level'], 0, 40), clamp(base['aqi'], 0, 200)

    stress_score = assign_risk_score(base)
    risk_label = score_to_label(stress_score)
    base['stress_score'], base['risk_label'], base['site_profile'] = round(stress_score, 2), risk_label, site_profile
    return base

profiles = ['wilderness', 'suburban', 'traffic', 'industrial', 'severe_mixed']
rows = [generate_realistic_site(p) for p in profiles for _ in range(500)]
df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)


In [ ]:
def calculate_diversity_score(row):
    growth_forms = np.array([
        row["crustose_percent"],
        row["foliose_percent"],
        row["fruticose_percent"]
    ]) / 100

    ecological_types = np.array([
        row["nitrophyte_abundance"],
        row["acidophyte_abundance"]
    ])

    ecological_types = ecological_types / (ecological_types.sum() + 1e-6)

    # Shannon diversity for growth forms
    growth_diversity = -np.sum(growth_forms * np.log(growth_forms + 1e-6))

    # Shannon diversity for ecological types
    eco_diversity = -np.sum(ecological_types * np.log(ecological_types + 1e-6))

    # Normalize approximate max:
    # max growth diversity = ln(3)
    # max ecological diversity = ln(2)
    growth_score = growth_diversity / np.log(3)
    eco_score = eco_diversity / np.log(2)

    diversity_score = (growth_score * 0.65 + eco_score * 0.35) * 100

    return np.clip(diversity_score, 0, 100)

# Apply the function to the existing dataframe
df["lichen_diversity_score"] = df.apply(calculate_diversity_score, axis=1)

# Display the new column alongside relevant data
display(df[['crustose_percent', 'foliose_percent', 'fruticose_percent', 'lichen_diversity_score']].head())

In [ ]:
df.to_csv("lichen_lens_synthetic_dataset.csv", index=False)
print("Synthetic dataset saved as lichen_lens_synthetic_dataset.csv")

In [ ]:
min_class_size = df["risk_label"].value_counts().min()

df = (
    df.groupby("risk_label", group_keys=False)
      .apply(lambda x: x.sample(min_class_size, random_state=42))
      .sample(frac=1, random_state=42)
      .reset_index(drop=True)
)

print(df["risk_label"].value_counts())

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

feature_columns = [
    "lichen_coverage_percent",
    "crustose_percent",
    "foliose_percent",
    "fruticose_percent",
    "nitrophyte_abundance",
    "acidophyte_abundance",
    "chlorosis_index",
    "iron_ppm",
    "copper_ppm",
    "lead_ppm",
    "zinc_ppm",
    "manganese_ppm",
    "no2_level",
    "so2_level",
    "aqi",
    "fluorine_estimate_ppb",
    "lichen_diversity_score",
    "crustose_fruticose_ratio",
    "nitrophyte_acidophyte_ratio",
    "traffic_metal_index",
    "industrial_metal_index",
    "total_metal_load",
    "lichen_health_index",
    "bio_stress_index",
    "chemical_stress_index"
]

target_column = "risk_label"

X = df[feature_columns]
y = df[target_column]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

X = df[feature_columns]
y = df[target_column]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

In [ ]:
leakage_columns = ["stress_score", "site_profile", "source_label"]

for col in leakage_columns:
    if col in feature_columns:
        raise ValueError(f"Data leakage: remove {col} from feature_columns")

print("No leakage columns in feature list.")

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import tensorflow as tf

num_features = X_train.shape[1]
num_classes = len(label_encoder.classes_)

model = Sequential([
    Dense(64, activation='relu', input_shape=(num_features,), kernel_regularizer=l2(0.0005)),
    BatchNormalization(),
    Dropout(0.12),

    Dense(48, activation='relu', kernel_regularizer=l2(0.0005)),
    BatchNormalization(),
    Dropout(0.10),

    Dense(24, activation='relu', kernel_regularizer=l2(0.0005)),

    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0007),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=25,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=8,
    min_lr=0.00003
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=220,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print("Lichen Lens Final Model Performance (with Diversity & Fluorine):")
print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_
))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=label_encoder.classes_
)

disp.plot(cmap="Blues")
plt.title("Confusion Matrix: Lichen Lens (25 Features)")
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

X = df[feature_columns]
y = df[target_column]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Lichen Lens Neural Network Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Lichen Lens Neural Network Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import numpy as np
import matplotlib.pyplot as plt

y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_
))

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=label_encoder.classes_
)

disp.plot()
plt.title("Lichen Lens Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score

y_pred_probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
final_acc = accuracy_score(y_test, y_pred)

print(f"Current Model Accuracy on Test Set: {final_acc * 100:.2f}%")

# Let's also check the count of correct vs incorrect predictions to see the impact of sample size
correct = (y_pred == y_test).sum()
total = len(y_test)
print(f"Correct predictions: {correct} out of {total}")

In [ ]:
if 'history' in globals():
    best_val_acc = max(history.history['val_accuracy'])
    print(f'Best Validation Accuracy: {best_val_acc * 100:.2f}%')
else:
    print('History object not found. Please ensure the training cell was executed successfully.')

In [ ]:
def pollution_fingerprint(site):
    site = site.iloc[0]

    fingerprints = []
    reasons = []

    if site["copper_ppm"] > 3 or site["zinc_ppm"] > 3:
        fingerprints.append("Traffic Signature")
        reasons.append("Elevated copper and zinc suggest brake pad wear and tire wear.")

    if site["iron_ppm"] > 3 or site["manganese_ppm"] > 3:
        fingerprints.append("Industrial Signature")
        reasons.append("Elevated iron and manganese suggest industrial or metal-processing pollution.")

    if site["lead_ppm"] > 0.5:
        fingerprints.append("Legacy Urban Dust")
        reasons.append("Detectable lead may suggest long-term urban dust accumulation.")

    if site["nitrophyte_abundance"] > site["acidophyte_abundance"] + 15:
        fingerprints.append("Nitrogen Stress")
        reasons.append("Nitrophytic lichens dominate, suggesting nitrogen-rich pollution conditions.")

    if site["lichen_coverage_percent"] < 20:
        fingerprints.append("Low Lichen Coverage")
        reasons.append("Low lichen coverage suggests poor environmental conditions for sensitive lichens.")

    if site["fruticose_percent"] < 15 and site["crustose_percent"] > 50:
        fingerprints.append("Pollution-Tolerant Growth Pattern")
        reasons.append("Crustose lichens dominate while fruticose lichens are rare, suggesting pollution stress.")

    if site["chlorosis_index"] > 0.35:
        fingerprints.append("Chlorosis / Ozone Stress")
        reasons.append("High chlorosis index suggests visible physiological stress or possible ozone damage.")

    if site["so2_level"] > 8:
        fingerprints.append("Sulfur Dioxide Stress")
        reasons.append("Elevated sulfur dioxide is associated with reduced lichen diversity.")

    if not fingerprints:
        fingerprints.append("Clean Baseline")
        reasons.append("The site does not show strong biological or chemical pollution stress signals.")

    return fingerprints, reasons

In [ ]:
def analyze_site(site_df):
    site_scaled = scaler.transform(site_df[feature_columns])
    probs = model.predict(site_scaled, verbose=0)[0]

    predicted_index = np.argmax(probs)
    predicted_label = label_encoder.inverse_transform([predicted_index])[0]
    confidence = probs[predicted_index]

    fingerprints, reasons = pollution_fingerprint(site_df)

    print("====== LICHEN LENS SITE REPORT ======")
    print("Predicted Air Quality Risk:", predicted_label)
    print("Confidence:", round(confidence * 100, 2), "%")

    print("\nClass Probabilities:")
    for label, prob in zip(label_encoder.classes_, probs):
        print(f"{label}: {round(prob * 100, 2)}%")

    print("\nPollution Fingerprint:")
    for item in fingerprints:
        print("-", item)

    print("\nScientific Explanation:")
    for reason in reasons:
        print("-", reason)

    print("\nSuggested Interpretation:")
    if predicted_label == "Severe":
        print("This site shows severe biological and chemical pollution stress. Lichens are sparse, damaged, or dominated by pollution-tolerant forms.")
    elif predicted_label == "High":
        print("This site shows strong pollution stress. Lichen biology and chemical indicators suggest reduced air quality.")
    elif predicted_label == "Moderate":
        print("This site shows moderate environmental stress. Continued monitoring is recommended.")
    else:
        print("This site appears to have healthier lichen conditions and lower pollution stress.")

In [ ]:
def prepare_site_features(site_df):
    """Calculates engineered features for a single-row site DataFrame."""
    df = site_df.copy()
    df['crustose_fruticose_ratio'] = df['crustose_percent'] / (df['fruticose_percent'] + 1)
    df['nitrophyte_acidophyte_ratio'] = df['nitrophyte_abundance'] / (df['acidophyte_abundance'] + 1)
    df['traffic_metal_index'] = (df['copper_ppm'] + df['zinc_ppm']) / 2
    df['industrial_metal_index'] = (df['iron_ppm'] + df['manganese_ppm']) / 2
    df['total_metal_load'] = df[['iron_ppm', 'copper_ppm', 'lead_ppm', 'zinc_ppm', 'manganese_ppm']].sum(axis=1)
    df['lichen_health_index'] = df['lichen_coverage_percent'] * 0.45 + df['fruticose_percent'] * 0.25 + df['acidophyte_abundance'] * 0.20 - df['chlorosis_index'] * 20
    df['bio_stress_index'] = (100 - df['lichen_coverage_percent']) * 0.30 + df['chlorosis_index'] * 35 + df['crustose_percent'] * 0.12 + (100 - df['fruticose_percent']) * 0.08
    df['chemical_stress_index'] = df['traffic_metal_index'] * 2.5 + df['industrial_metal_index'] * 2.2 + df['lead_ppm'] * 8 + df['so2_level'] * 0.7 + df['no2_level'] * 0.4
    return df

# Re-apply full engineering to the main dataframe
df = prepare_site_features(df)
print("Feature Engineering Updated.")
print("Columns:", df.columns.tolist())

In [ ]:
traffic_site = pd.DataFrame([{
    "lichen_coverage_percent": 14,
    "crustose_percent": 72,
    "foliose_percent": 20,
    "fruticose_percent": 8,
    "nitrophyte_abundance": 82,
    "acidophyte_abundance": 14,
    "chlorosis_index": 0.44,
    "iron_ppm": 1.1,
    "copper_ppm": 4.8,
    "lead_ppm": 0.42,
    "zinc_ppm": 5.4,
    "manganese_ppm": 1.0,
    "no2_level": 28,
    "so2_level": 4,
    "aqi": 78,
    # Added missing features required by the model
    "fluorine_estimate_ppb": 0.18,
    "lichen_diversity_score": 25.0
}])

traffic_site_prepared = prepare_site_features(traffic_site)
analyze_site(traffic_site_prepared)

In [ ]:
industrial_site = pd.DataFrame([{
    "lichen_coverage_percent": 12,
    "crustose_percent": 78,
    "foliose_percent": 17,
    "fruticose_percent": 5,
    "nitrophyte_abundance": 64,
    "acidophyte_abundance": 22,
    "chlorosis_index": 0.48,
    "iron_ppm": 5.6,
    "copper_ppm": 1.6,
    "lead_ppm": 0.51,
    "zinc_ppm": 1.9,
    "manganese_ppm": 4.9,
    "no2_level": 21,
    "so2_level": 9,
    "aqi": 88,
    "fluorine_estimate_ppb": 0.45,
    "lichen_diversity_score": 22.0
}])

industrial_site_prepared = prepare_site_features(industrial_site)
analyze_site(industrial_site_prepared)

In [ ]:
severe_site = pd.DataFrame([{
    "lichen_coverage_percent": 3,
    "crustose_percent": 88,
    "foliose_percent": 10,
    "fruticose_percent": 2,
    "nitrophyte_abundance": 85,
    "acidophyte_abundance": 8,
    "chlorosis_index": 0.68,
    "iron_ppm": 6.4,
    "copper_ppm": 4.2,
    "lead_ppm": 0.82,
    "zinc_ppm": 5.7,
    "manganese_ppm": 5.5,
    "no2_level": 35,
    "so2_level": 22,
    "aqi": 130,
    "fluorine_estimate_ppb": 0.85,
    "lichen_diversity_score": 12.0
}])

severe_site_prepared = prepare_site_features(severe_site)
analyze_site(severe_site_prepared)

In [ ]:
suburban_site = pd.DataFrame([{
    "lichen_coverage_percent": 48,
    "crustose_percent": 42,
    "foliose_percent": 35,
    "fruticose_percent": 23,
    "nitrophyte_abundance": 45,
    "acidophyte_abundance": 42,
    "chlorosis_index": 0.25,
    "iron_ppm": 0.8,
    "copper_ppm": 1.1,
    "lead_ppm": 0.12,
    "zinc_ppm": 1.3,
    "manganese_ppm": 0.7,
    "no2_level": 14,
    "so2_level": 3,
    "aqi": 52,
    "fluorine_estimate_ppb": 0.10,
    "lichen_diversity_score": 65.0
}])

suburban_site_prepared = prepare_site_features(suburban_site)
analyze_site(suburban_site_prepared)

In [ ]:
def sensitivity_analysis(site_df, change_percent=0.15):
    original_scaled = scaler.transform(site_df[feature_columns])
    original_probs = model.predict(original_scaled, verbose=0)[0]
    original_class = np.argmax(original_probs)
    original_confidence = original_probs[original_class]

    impacts = []

    for feature in feature_columns:
        modified_site = site_df.copy()
        original_value = modified_site.loc[0, feature]
        modified_site.loc[0, feature] = original_value * (1 + change_percent)

        modified_scaled = scaler.transform(modified_site[feature_columns])
        modified_probs = model.predict(modified_scaled, verbose=0)[0]
        modified_confidence = modified_probs[original_class]

        impact = abs(original_confidence - modified_confidence)

        impacts.append({
            "feature": feature,
            "impact": impact
        })

    impact_df = pd.DataFrame(impacts).sort_values(by="impact", ascending=False)
    return impact_df

# Run sensitivity analysis for a representative of each profile
profiles = ['wilderness', 'suburban', 'traffic', 'industrial', 'severe_mixed']
profile_results = {}

for profile in profiles:
    print(f"Processing sensitivity for: {profile}...")
    # Get the first occurrence of each profile from the original dataframe
    sample = df[df['site_profile'] == profile].head(1).copy().reset_index(drop=True)
    sample_prepared = prepare_site_features(sample)
    profile_results[profile] = sensitivity_analysis(sample_prepared)

# Display the top features for each to verify
for profile, result in profile_results.items():
    print(f"\nTop 5 Sensitivity Factors for {profile.upper()}:")
    print(result.head(5))

In [ ]:
def plot_detailed_individual_sensitivities(results_dict):
    profiles = list(results_dict.keys())

    for profile in profiles:
        result = results_dict[profile] # This contains all 23 features

        plt.figure(figsize=(12, 10))
        colors = plt.cm.plasma(np.linspace(0, 0.8, len(result)))

        plt.barh(result["feature"], result["impact"], color=colors)

        plt.title(f"Full Feature Sensitivity: {profile.upper()} Profile", fontsize=16, fontweight='bold')
        plt.xlabel("Confidence Impact (Delta)", fontsize=12)
        plt.ylabel("Feature", fontsize=12)

        plt.gca().invert_yaxis()
        plt.grid(axis='x', linestyle='--', alpha=0.4)
        plt.tight_layout()
        plt.show()
        print("\n" + "="*80 + "\n")

# Generate the large separate plots for each profile
plot_detailed_individual_sensitivities(profile_results)

In [ ]:
import seaborn as sns

# 1. Feature Correlation Heatmap
plt.figure(figsize=(16, 12))
corr_matrix = df[feature_columns].corr()
sns.heatmap(corr_matrix, annot=False, cmap='RdBu_r', center=0)
plt.title("Correlation Matrix: Biological and Chemical Markers", fontsize=16)
plt.show()

In [ ]:
from sklearn.decomposition import PCA

# 2. PCA Clustering of Environmental Profiles
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(12, 8))
sns.scatterplot(
    x=X_pca[:, 0], y=X_pca[:, 1],
    hue=df['site_profile'],
    palette='viridis',
    alpha=0.6
)
plt.title("PCA: Clustering of Environmental Profiles (Lichen Lens Data)", fontsize=16)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Site Profile", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import joblib

# Save the Keras Neural Network in the native Keras format
model.save('lichen_lens_model.keras')

# Save the scaling and encoding artifacts needed for deployment
joblib.dump(scaler, 'lichen_lens_scaler.pkl')
joblib.dump(label_encoder, 'lichen_lens_label_encoder.pkl')

print("Deployment artifacts saved:")
print("- lichen_lens_model.keras (Native Keras format)")
print("- lichen_lens_scaler.pkl (Feature Normalization)")
print("- lichen_lens_label_encoder.pkl (Target Class Mapping)")

In [ ]:
from google.colab import files

# Download the native Keras model
files.download('lichen_lens_model.keras')

# Download the preprocessing artifacts
files.download('lichen_lens_scaler.pkl')
files.download('lichen_lens_label_encoder.pkl')

print("Download requests sent for all 3 artifacts.")